Lectura del archivo y normalizacion del archivo y lecutra de los modulos de la automatizacion


In [6]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
import sys
import os
import pandas as pd
from pathlib import Path
import plotly.express as px



# Detectar raíz del proyecto (subiendo desde el cwd)
project_root = Path(os.getcwd()).resolve().parent  # sube un nivel desde notebooks/
sys.path.append(str(project_root))

# Ruta hacia el módulo src/
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

# Importar módulos desde src/
from src.data_loading import (
    cargar_datos_excel
)
from src.data_cleaning import (
    estandarizar_columnas,
    convertir_fechas,
)
from src.logistica import (
    calcular_tiempo_entrega,    
)
from src.data_cleaning import (
    limpiar_metodo_envio
)
from src.exportacion import (
    exportar_excel,
)




In [8]:
ruta_excel = "../data/raw/Sales report completa.xlsx" 
df = (
    cargar_datos_excel(ruta_excel, sheet_name="Ventas Supermercado")
    .pipe(estandarizar_columnas)
    .pipe(convertir_fechas)
    .pipe(calcular_tiempo_entrega)
    .pipe(limpiar_metodo_envio)
)

visualizacion de las columnas utilizando jupyter

In [10]:
pd.DataFrame(df.columns, columns=['Columnas'])


,Columnas
0,numero_venta
1,mes_salida
2,dia_salida
3,ano_salida
4,mes_entrega
5,dia_entrega
6,ano_entrega
7,metodo_envio
8,numero_cliente
9,nombre_cliente


medias estadisticas para analisis del tiempo de entrega

In [11]:
df['tiempo_entrega'].describe()


count    51290.000000
mean         3.969370
std          1.729437
min          0.000000
25%          3.000000
50%          4.000000
75%          5.000000
max          7.000000
Name: tiempo_entrega, dtype: float64

inicio de analisis geografico

In [18]:
# Contar número de envíos por ciudad
top_ciudades = df.groupby(['pais', 'ciudad']).size().reset_index(name='num_envios')

# Ordenar de mayor a menor
top_ciudades = top_ciudades.sort_values(by='num_envios', ascending=False)

# Mostrar primeras filas
print(top_ciudades.head(20))


                    pais         ciudad  num_envios
3427       United States  New York City         915
3364       United States    Los Angeles         747
3472       United States   Philadelphia         537
3536       United States  San Francisco         510
915   Dominican Republic  Santo Domingo         443
2437         Philippines         Manila         432
3550       United States        Seattle         428
3305       United States        Houston         377
1561            Honduras    Tegucigalpa         362
1726           Indonesia        Jakarta         337
2331           Nicaragua        Managua         336
2360             Nigeria          Lagos         333
3178       United States        Chicago         314
2833              Turkey       Istanbul         314
2144              Mexico    Mexico City         300
2776            Thailand        Bangkok         287
124            Australia         Sydney         271
3028      United Kingdom         London         257
932         

In [21]:
# Tomar solo las 20 ciudades con más envíos
top5_ciudades = top_ciudades.head(20)


In [24]:

fig1 = px.bar(
    top5_ciudades,
    x='num_envios',
    y='ciudad',
    color='pais',
    orientation='h',
    title='Top 20 ciudades con más envíos',
    labels={'num_envios': 'Número de envíos', 'ciudad': 'Ciudad'}
)

fig1.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    plot_bgcolor='white'
)

fig1.show()


In [ ]:
import plotly.express as px

fig5 = px.box(
    df,
    x='metodo_envio',
    y='tiempo_entrega',
    color='metodo_envio',
    title='Duración del envío por Método de Envío',
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig5.update_layout(
    xaxis_title='Método de Envío',
    yaxis_title='Tiempo de Entrega',
    boxmode='group'
)

fig5.show()


In [ ]:
import plotly.express as px

fig3 = px.box(
    df,
    x='prioridad_envio',
    y='tiempo_entrega',
    color='prioridad_envio',
    title='Duración del envío por Prioridad de Envío',
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig3.update_layout(
    xaxis_title='Prioridad de Envío',
    yaxis_title='Tiempo de Entrega',
    boxmode='group'
)

fig3.show()


In [ ]:
fig4 = px.violin(
    df,
    x='prioridad_envio',
    y='tiempo_entrega',
    color='prioridad_envio',
    box=True,
    points='all',
    title='Distribución del tiempo de entrega por Prioridad de Envío',
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig4.update_layout(
    xaxis_title='Prioridad de Envío',
    yaxis_title='Tiempo de Entrega'
)

fig4.show()


In [ ]:
fig5.write_html("duracion_metodo_envio.html")
